***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 5 章：成像](5_0_introduction.ipynb)
    * 上一章：[第 4 章：可见度空间](../4_Visibility_Space/4_0_introduction.ipynb)
    * 下一节：[5.1 空间频率](5_1_spatial_frequencies.ipynb)

***


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML
%matplotlib inline
HTML('../style/course.css') #apply general CSS


导入本节所需的专用模块。


# 第 5 章：成像<a id='imaging:sec:intro'></a>

第 4 章说明了干涉阵直接测得的是可见度，而不是天空图像中的像素值。第 5 章开始讨论下一步：怎样把分散在 `uv` 平面上的可见度样本重新组织成图像。若可见度函数在整个平面上连续已知，按照 van Cittert-Zernike 定理，只需一次二维反 Fourier 变换就能得到表观天空亮度分布；但真实阵列只在有限的基线、时间和频率上给出离散采样，因此成像从一开始就是一个带有采样缺口的逆问题。

这一点决定了射电干涉成像与普通照相成像的差别。相机是在像平面上直接积分，模糊主要来自光学系统的点扩散函数；干涉阵则先在空间频率域中采样，再通过数学变换回到图像域。于是，图像质量不仅取决于天线口径和基线长度，还取决于 `uv` 覆盖、加权方式、规则网格化误差以及小视场近似是否成立。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

n = 192
x = np.linspace(-1.0, 1.0, n)
X, Y = np.meshgrid(x, x)


def gaussian(x0, y0, sx, sy, amp):
    return amp * np.exp(-0.5 * (((X - x0) / sx) ** 2 + ((Y - y0) / sy) ** 2))


sky = (
    gaussian(-0.35, 0.25, 0.10, 0.08, 1.2)
    + gaussian(0.20, -0.10, 0.18, 0.12, 0.9)
    + gaussian(0.45, 0.35, 0.06, 0.06, 0.7)
)
sky /= sky.max()

rng = np.random.default_rng(12)
uv_mask = np.zeros((n, n), dtype=float)
coords = []
for _ in range(180):
    radius = rng.uniform(0.10, 0.92)
    angle = rng.uniform(0.0, np.pi)
    u = radius * np.cos(angle)
    v = 0.55 * radius * np.sin(angle)
    i = int(np.clip((v + 1.0) * 0.5 * (n - 1), 0, n - 1))
    j = int(np.clip((u + 1.0) * 0.5 * (n - 1), 0, n - 1))
    coords.append((i, j))
    coords.append((n - 1 - i, n - 1 - j))
for i, j in coords:
    uv_mask[i, j] = 1.0
uv_mask[n // 2, n // 2] = 1.0

sky_ft = np.fft.fftshift(np.fft.fft2(sky))
dirty_image = np.fft.ifft2(np.fft.ifftshift(sky_ft * uv_mask)).real
dirty_image /= np.max(np.abs(dirty_image))

dirty_beam = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(uv_mask))).real
dirty_beam /= np.max(np.abs(dirty_beam))

clean_beam = np.exp(-0.5 * ((X / 0.09) ** 2 + (Y / 0.09) ** 2))
clean_beam /= clean_beam.sum()
restored = np.fft.ifft2(np.fft.fft2(sky) * np.fft.fft2(np.fft.ifftshift(clean_beam))).real
restored /= restored.max()

fig, axes = plt.subplots(1, 4, figsize=(15.5, 3.8))

axes[0].imshow(sky, origin='lower', cmap='magma')
axes[0].set_title('sky brightness')
axes[0].set_xticks([])
axes[0].set_yticks([])

uv_y, uv_x = np.nonzero(uv_mask)
axes[1].scatter((uv_x - n / 2) / (n / 2), (uv_y - n / 2) / (n / 2), s=8, c='#1d3557')
axes[1].set_title('sampled visibilities')
axes[1].set_xlabel(r'$u$')
axes[1].set_ylabel(r'$v$')
axes[1].set_aspect('equal')
axes[1].grid(alpha=0.25)
axes[1].set_xlim(-1.05, 1.05)
axes[1].set_ylim(-1.05, 1.05)

axes[2].imshow(dirty_image, origin='lower', cmap='magma')
axes[2].contour(dirty_beam, levels=[0.25, 0.5], colors='white', linewidths=0.7)
axes[2].set_title('dirty image')
axes[2].set_xticks([])
axes[2].set_yticks([])

axes[3].imshow(restored, origin='lower', cmap='magma')
axes[3].set_title('restored image')
axes[3].set_xticks([])
axes[3].set_yticks([])

for xpos, label in [(0.255, 'Fourier sampling'), (0.505, 'inverse transform'), (0.755, 'deconvolution')]:
    fig.text(xpos, 0.92, label, ha='center', va='center', fontsize=11)

fig.tight_layout(rect=[0, 0, 1, 0.88])
fig.savefig(fig_dir / 'chapter5_imaging_flow.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![成像流程示意图](figures/chapter5_imaging_flow.png)

**图 5.0.1** 从天空亮度到射电干涉图像的基本流程。连续天空先被投影为有限的可见度采样，再经反 Fourier 变换形成脏图像，最后通过去卷积和恢复步骤得到可解释的成像结果。


本章的主线可以概括为一句话：**成像就是把不完备的 Fourier 采样转换成尽可能接近真实天空的亮度分布**。为了把这句话讲清，需要依次回答四个问题。

首先，空间频率到底对应图像中的什么结构，为什么低频控制大尺度结构而高频控制边缘和细节。其次，离散采样的可见度为什么会在图像域中表现为脏波束卷积，以及脏图像与真实天空之间究竟是什么关系。再次，为什么实际计算不能直接对不规则采样点做快速 Fourier 变换，而必须先进行网格化和反网格化。最后，在大视场、非共面基线和明显的 `w` 项存在时，二维 Fourier 近似又会在何处失效。

第 5.1 节先从空间频率的角度建立直觉，并通过普通图像、点源和双点源三个层次说明幅度、相位与结构之间的关系。第 5.2 节把离散采样写成采样函数，并由此引出脏波束与点扩散函数。第 5.3 节讨论可见度如何从不规则采样点映射到规则网格，这是 FFT 成像的数值基础。第 5.4 节进一步说明自然权重、均匀权重和鲁棒权重怎样在灵敏度与分辨率之间取舍。第 5.5 节则回到第 4 章末尾提出的问题，讨论宽场成像中 `w` 项带来的误差及其修正。第 5.6 节把前面这些原理综合为成像参数选择的物理依据，说明 `cell`、`imsize`、权重、taper、gridder、deconvolver、阈值和 mask 如何共同服务科学目标。


### 章节大纲

1. [5.1 空间频率](5_1_spatial_frequencies.ipynb)
2. [5.2 采样函数与点扩散函数](5_2_sampling_functions_and_psfs.ipynb)
3. [5.3 用于 FFT 的网格化与反网格化](5_3_gridding_and_degridding.ipynb)
4. [5.4 脏图像与可见度权重](5_4_imaging_weights.ipynb)
5. [5.5 小角近似的失效与 $w$ 项](5_5_widefield_effect.ipynb)
6. [5.6 成像参数选择的物理依据](5_6_imaging_parameter_decisions.ipynb)
7. [5.A 匹配滤波](5_A_matched_filter.ipynb)
8. [5.x 延伸阅读与参考资料](5_x_further_reading_and_references.ipynb)


***

* 下一节：[5.1 空间频率](5_1_spatial_frequencies.ipynb)
